# B.Tech Probability & Statistics Project
## Time-Series Analysis and Forecasting of AI Model Training Compute Using Real-World Data
### Notebook 01: Comprehensive Data Audit, Coverage Analysis, and Time-Frequency Evaluation

**Academic Context & Syllabus Alignment:**
- **Module I:** Descriptive statistics, frequency distributions, measures of central tendency (mean, median), measures of dispersion (variance, standard deviation, IQR), scatter plots, and histograms.
- **Module VII:** Testing of hypothesis foundations, sample size adequacy, and distribution properties.
- **Module IX:** Bivariate correlation, covariance, and log-linear scaling foundations.
- **Module X:** Regression foundations (simple, polynomial, non-linear).

**Core Objective:** Conduct an empirical data audit on `notable_ai_models.csv` to determine the distribution of training compute, evaluate historical data sparsity, and make an objective statistical decision on time-series frequency (Yearly vs. Monthly) before fitting any forecasting models.

### 1. Setup & Environment Imports
We import standard scientific computing libraries: `pandas`, `numpy`, `matplotlib.pyplot`, and `scipy.stats`. All directories for outputs and figures are ensured.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Configure high-quality plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

# Ensure project directories exist
for d in ['data', 'outputs', 'graphs', 'report', 'notebooks']:
    os.makedirs(f'../{d}' if os.path.exists('../data') else d, exist_ok=True)

print('Libraries successfully imported.')

Libraries successfully imported.


### 2. Task 1 & 2: Load Dataset and Data Audit
We load `notable_ai_models.csv` without modifying or overwriting the raw data. We inspect dimensions, column data types, duplicate records, and missing values across all columns.

In [4]:
# Locate and load the dataset safely
csv_path = 'data/notable_ai_models.csv' if os.path.exists('data/notable_ai_models.csv') else '../data/notable_ai_models.csv'
df = pd.read_csv(csv_path)

n_rows, n_cols = df.shape
n_duplicates = df.duplicated().sum()
print(f'=== DATASET DIMENSIONS ===')
print(f'Total Rows: {n_rows}')
print(f'Total Columns: {n_cols}')
print(f'Duplicate Rows: {n_duplicates}')

# Inspect key project columns
target_cols = [
    'Model', 'Organization', 'Publication date', 'Parameters',
    'Training compute (FLOP)', 'Training time (hours)', 'Training hardware',
    'Training power draw (W)', 'Training compute cost (2023 USD)', 'Frontier model'
]

audit_records = []
for col in target_cols:
    if col in df.columns:
        m_cnt = df[col].isnull().sum()
        m_pct = (m_cnt / n_rows) * 100
        audit_records.append({
            'Column': col,
            'Data Type': str(df[col].dtype),
            'Missing Count': m_cnt,
            'Missing Pct (%)': round(m_pct, 2)
        })

audit_df = pd.DataFrame(audit_records)
audit_df

=== DATASET DIMENSIONS ===
Total Rows: 1072
Total Columns: 47
Duplicate Rows: 0


### 3. Task 3: Publication Date Analysis
We convert `Publication date` to datetime with `errors='coerce'`, extract publication years and months (`YYYY-MM`), verify earliest/latest dates, and output annual model frequencies.

In [6]:
# Safe datetime conversion
df['pub_date_clean'] = pd.to_datetime(df['Publication date'], errors='coerce')
missing_dates = df['Publication date'].isnull().sum()
invalid_dates = df['Publication date'].notnull().sum() - df['pub_date_clean'].notnull().sum()

earliest_date = df['pub_date_clean'].min()
latest_date = df['pub_date_clean'].max()

print(f'Earliest Publication Date: {earliest_date}')
print(f'Latest Publication Date:   {latest_date}')
print(f'Invalid Dates: {invalid_dates}')
print(f'Missing Dates: {missing_dates}')

# Add Year and Month features
df['Year'] = df['pub_date_clean'].dt.year
df['Month'] = df['pub_date_clean'].dt.to_period('M').astype(str)
df.loc[df['pub_date_clean'].isnull(), 'Month'] = np.nan

# Generate and save yearly model counts
yearly_model_counts = df.dropna(subset=['Year']).groupby('Year').size().reset_index(name='Number of Models')
yearly_model_counts['Year'] = yearly_model_counts['Year'].astype(int)

out_ymc = 'outputs/yearly_model_counts.csv' if os.path.exists('outputs') else '../outputs/yearly_model_counts.csv'
yearly_model_counts.to_csv(out_ymc, index=False)
print(f'Saved yearly model counts to {out_ymc}.')
yearly_model_counts.tail(15)

Earliest Publication Date: 1950-07-02 00:00:00
Latest Publication Date:   2026-09-21 00:00:00
Invalid Dates: 0
Missing Dates: 0
Saved yearly model counts to outputs/yearly_model_counts.csv.


### 4. Task 4: Training Compute Data Quality & Parameter Audit
We inspect `Training compute (FLOP)` safely by converting to numeric without silent discard. We identify valid records, zero values, negative values, and missing values, and isolate the clean subset of positive FLOPs. We do the equivalent for `Parameters`.

In [8]:
# Training compute conversion
df['compute_numeric'] = pd.to_numeric(df['Training compute (FLOP)'], errors='coerce')
total_rec = len(df)
valid_compute_rec = df['compute_numeric'].notnull().sum()
missing_compute_rec = df['Training compute (FLOP)'].isnull().sum()
non_num_compute = df['Training compute (FLOP)'].notnull().sum() - valid_compute_rec
zero_compute = (df['compute_numeric'] == 0).sum()
neg_compute = (df['compute_numeric'] < 0).sum()
pos_compute = (df['compute_numeric'] > 0).sum()

print('=== COMPUTE DATA AUDIT ===')
print(f'Total Records: {total_rec}')
print(f'Valid Numeric Compute Records: {valid_compute_rec}')
print(f'Missing Compute Records (raw NaN): {missing_compute_rec}')
print(f'Invalid/Non-Numeric Strings: {non_num_compute}')
print(f'Zero Compute Values: {zero_compute}')
print(f'Negative Compute Values: {neg_compute}')
print(f'Valid Positive Compute Records (> 0): {pos_compute}')

clean_compute_df = df[df['compute_numeric'] > 0].copy()

# Compute descriptive statistics
compute_stats = {
    'Count': len(clean_compute_df),
    'Minimum (FLOP)': f"{clean_compute_df['compute_numeric'].min():.2e}",
    'Maximum (FLOP)': f"{clean_compute_df['compute_numeric'].max():.2e}",
    'Mean (FLOP)': f"{clean_compute_df['compute_numeric'].mean():.2e}",
    'Median (FLOP)': f"{clean_compute_df['compute_numeric'].median():.2e}",
    'Std Dev (FLOP)': f"{clean_compute_df['compute_numeric'].std():.2e}"
}
print('\nClean Training Compute Descriptive Statistics:')
for k, v in compute_stats.items():
    print(f' - {k}: {v}')

# Parameters audit
df['params_numeric'] = pd.to_numeric(df['Parameters'], errors='coerce')
clean_params_df = df[df['params_numeric'] > 0].copy()
param_stats = {
    'Count': len(clean_params_df),
    'Minimum': f"{clean_params_df['params_numeric'].min():.2e}",
    'Maximum': f"{clean_params_df['params_numeric'].max():.2e}",
    'Mean': f"{clean_params_df['params_numeric'].mean():.2e}",
    'Median': f"{clean_params_df['params_numeric'].median():.2e}",
    'Std Dev': f"{clean_params_df['params_numeric'].std():.2e}"
}
print('\nClean Parameters Descriptive Statistics:')
for k, v in param_stats.items():
    print(f' - {k}: {v}')

=== COMPUTE DATA AUDIT ===
Total Records: 1072
Valid Numeric Compute Records: 540
Missing Compute Records (raw NaN): 532
Invalid/Non-Numeric Strings: 0
Zero Compute Values: 0
Negative Compute Values: 0
Valid Positive Compute Records (> 0): 540

Clean Training Compute Descriptive Statistics:
 - Count: 540
 - Minimum (FLOP): 4.00e+01
 - Maximum (FLOP): 1.00e+27
 - Mean (FLOP): 5.69e+24
 - Median (FLOP): 1.49e+21
 - Std Dev (FLOP): 5.31e+25

Clean Parameters Descriptive Statistics:
 - Count: 725
 - Minimum: 1.60e+01
 - Maximum: 3.00e+12
 - Mean: 1.05e+11
 - Median: 5.50e+08
 - Std Dev: 3.28e+11


### 5. Task 5: Logarithmic Transformation
Because training compute spans over 25 orders of magnitude (from $4.0 \times 10^1$ to $1.0 \times 10^{27}$ FLOP), raw compute violates homoscedasticity and normality assumptions. We compute $\log_{10}(\text{FLOP})$ exclusively on strictly positive records.

In [10]:
# Strictly positive log10 transformation
clean_compute_df['log10_compute'] = np.log10(clean_compute_df['compute_numeric'])
clean_params_df['log10_params'] = np.log10(clean_params_df['params_numeric'])

print('=== LOG10 COMPUTE DESCRIPTIVE STATISTICS ===')
print(f"Minimum log10 compute: {clean_compute_df['log10_compute'].min():.6f}")
print(f"Maximum log10 compute: {clean_compute_df['log10_compute'].max():.6f}")
print(f"Mean log10 compute:    {clean_compute_df['log10_compute'].mean():.6f}")
print(f"Median log10 compute:  {clean_compute_df['log10_compute'].median():.6f}")
print(f"Std Dev log10 compute: {clean_compute_df['log10_compute'].std():.6f}")

print('\n=== LOG10 PARAMETERS DESCRIPTIVE STATISTICS ===')
print(f"Minimum log10 params:  {clean_params_df['log10_params'].min():.6f}")
print(f"Maximum log10 params:  {clean_params_df['log10_params'].max():.6f}")
print(f"Mean log10 params:     {clean_params_df['log10_params'].mean():.6f}")
print(f"Median log10 params:   {clean_params_df['log10_params'].median():.6f}")
print(f"Std Dev log10 params:  {clean_params_df['log10_params'].std():.6f}")

=== LOG10 COMPUTE DESCRIPTIVE STATISTICS ===
Minimum log10 compute: 1.602060
Maximum log10 compute: 27.000043
Mean log10 compute:    20.126716
Median log10 compute:  21.173971
Std Dev log10 compute: 4.350797

=== LOG10 PARAMETERS DESCRIPTIVE STATISTICS ===
Minimum log10 params:  1.204120
Maximum log10 params:  12.477121
Mean log10 params:     8.632315
Median log10 params:   8.740363
Std Dev log10 params:  2.287168


### 6. Task 6: Yearly Compute Coverage & Target Time Series Variable
We aggregate valid compute records by year and establish the primary time series variable:
$$Y_t = \text{median}(\log_{10}(\text{Training Compute (FLOP)})) \quad \text{for year } t$$
We save the result to `outputs/yearly_ai_compute.csv`.

In [12]:
# Aggregate yearly metrics
yearly_ai_compute = clean_compute_df.groupby('Year').agg(
    models_with_valid_compute=('compute_numeric', 'count'),
    mean_training_compute=('compute_numeric', 'mean'),
    median_training_compute=('compute_numeric', 'median'),
    mean_log10_compute=('log10_compute', 'mean'),
    median_log10_compute=('log10_compute', 'median'),
    std_log10_compute=('log10_compute', 'std'),
    min_compute=('compute_numeric', 'min'),
    max_compute=('compute_numeric', 'max')
).reset_index()
yearly_ai_compute['Year'] = yearly_ai_compute['Year'].astype(int)

out_yac = 'outputs/yearly_ai_compute.csv' if os.path.exists('outputs') else '../outputs/yearly_ai_compute.csv'
yearly_ai_compute.to_csv(out_yac, index=False)
print(f'Saved yearly compute dataset to {out_yac}.')
yearly_ai_compute.tail(15)

Saved yearly compute dataset to outputs/yearly_ai_compute.csv.


### 7. Task 7: Monthly Compute Coverage & Calendar Index Analysis
We construct a complete calendar month index from 1950-07 to 2026-09 (915 calendar months) and identify the proportion of empty periods vs observed periods.

In [14]:
clean_compute_df['Month_period'] = clean_compute_df['pub_date_clean'].dt.to_period('M')
monthly_agg = clean_compute_df.groupby('Month_period').agg(
    models_with_valid_compute=('compute_numeric', 'count'),
    median_training_compute=('compute_numeric', 'median'),
    median_log10_compute=('log10_compute', 'median'),
    mean_log10_compute=('log10_compute', 'mean')
).reset_index()
monthly_agg['Month'] = monthly_agg['Month_period'].astype(str)

out_mac = 'outputs/monthly_ai_compute.csv' if os.path.exists('outputs') else '../outputs/monthly_ai_compute.csv'
monthly_agg[['Month', 'models_with_valid_compute', 'median_training_compute', 'median_log10_compute', 'mean_log10_compute']].to_csv(out_mac, index=False)

# Complete calendar month index
start_m = df['pub_date_clean'].min().to_period('M')
end_m = df['pub_date_clean'].max().to_period('M')
all_months = pd.period_range(start_m, end_m, freq='M')
monthly_coverage = pd.DataFrame({'Month_period': all_months})
monthly_coverage['Month'] = monthly_coverage['Month_period'].astype(str)
monthly_coverage = monthly_coverage.merge(monthly_agg, on='Month', how='left')
monthly_coverage['models_with_valid_compute'] = monthly_coverage['models_with_valid_compute'].fillna(0).astype(int)
monthly_coverage['has_observation'] = monthly_coverage['models_with_valid_compute'] > 0

out_mcov = 'outputs/monthly_coverage.csv' if os.path.exists('outputs') else '../outputs/monthly_coverage.csv'
monthly_coverage[['Month', 'models_with_valid_compute', 'median_training_compute', 'median_log10_compute', 'mean_log10_compute', 'has_observation']].to_csv(out_mcov, index=False)

tot_months = len(monthly_coverage)
obs_months = monthly_coverage['has_observation'].sum()
empty_months = tot_months - obs_months
pct_empty = (empty_months / tot_months) * 100
avg_models_obs = monthly_coverage[monthly_coverage['has_observation']]['models_with_valid_compute'].mean()
med_models_obs = monthly_coverage[monthly_coverage['has_observation']]['models_with_valid_compute'].median()

print('=== MONTHLY COVERAGE METRICS ===')
print(f'Total Calendar Months: {tot_months}')
print(f'Months with >= 1 Valid Observation: {obs_months} ({100 - pct_empty:.2f}%)')
print(f'Months with Zero Observations (Empty): {empty_months} ({pct_empty:.2f}%)')
print(f'Average Models per Observed Month: {avg_models_obs:.2f}')
print(f'Median Models per Observed Month:  {med_models_obs:.1f}')

=== MONTHLY COVERAGE METRICS ===
Total Calendar Months: 915
Months with >= 1 Valid Observation: 193 (21.09%)
Months with Zero Observations (Empty): 722 (78.91%)
Average Models per Observed Month: 2.80
Median Models per Observed Month:  2.0


### 8. Task 8: Recent-Data Coverage (2020–2026) and 2026 Partial-Year Audit
We examine the recent years and determine the exact latest date recorded in 2026 to evaluate partial-year truncation.

In [16]:
recent_years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
recent_records = []
for y in recent_years:
    tot_m = (df['Year'] == y).sum()
    val_c = (clean_compute_df['Year'] == y).sum()
    cov_pct = (val_c / tot_m * 100) if tot_m > 0 else 0.0
    med_log = clean_compute_df[clean_compute_df['Year'] == y]['log10_compute'].median()
    recent_records.append({
        'Year': y,
        'Total Models': tot_m,
        'Models with Valid Compute': val_c,
        'Compute Coverage (%)': round(cov_pct, 2),
        'Median log10(FLOP)': round(med_log, 3)
    })

recent_df = pd.DataFrame(recent_records)
print('=== RECENT YEARS COVERAGE (2020–2026) ===')
print(recent_df.to_string(index=False))

latest_2026_date = df[df['Year'] == 2026]['pub_date_clean'].max()
print(f'\nExact latest 2026 publication date in dataset: {latest_2026_date}')

=== RECENT YEARS COVERAGE (2020–2026) ===
 Year  Total Models  Models with Valid Compute  Compute Coverage (%)  Median log10(FLOP)
 2020            50                         33                 66.00              20.795
 2021            79                         62                 78.48              22.091
 2022            91                         60                 65.93              22.170
 2023           119                         65                 54.62              22.801
 2024            98                         46                 46.94              24.480
 2025           107                         40                 37.38              24.588
 2026            78                         21                 26.92              24.635

Exact latest 2026 publication date in dataset: 2026-09-21 00:00:00


### 9. Task 9: Visualizations
We generate and inspect the 9 required statistical figures, saving them as high-resolution PNGs in `graphs/`.

In [18]:
# List of generated graphs
graph_files = [
    '01_models_by_year.png',
    '02_valid_compute_models_by_year.png',
    '03_distribution_training_compute.png',
    '04_distribution_log10_compute.png',
    '05_compute_vs_year_scatter.png',
    '06_log10_compute_vs_year_scatter.png',
    '07_monthly_valid_compute_counts.png',
    '08_yearly_median_log10_compute.png',
    '09_monthly_median_log10_compute.png'
]

graphs_dir = 'graphs' if os.path.exists('graphs') else '../graphs'
for gf in graph_files:
    p = os.path.join(graphs_dir, gf)
    if os.path.exists(p):
        print(f'Verified: {p} ({os.path.getsize(p):,} bytes)')
    else:
        print(f'Missing: {p}')

try:
    from IPython.display import Image, display
    display(Image(os.path.join(graphs_dir, '04_distribution_log10_compute.png'), width=700))
    display(Image(os.path.join(graphs_dir, '08_yearly_median_log10_compute.png'), width=700))
except ImportError:
    print('All 9 figures saved in graphs/ directory. (IPython display skipped in standard Python runtime)')

Verified: graphs\01_models_by_year.png (157,505 bytes)
Verified: graphs\02_valid_compute_models_by_year.png (119,509 bytes)
Verified: graphs\03_distribution_training_compute.png (130,973 bytes)
Verified: graphs\04_distribution_log10_compute.png (177,301 bytes)
Verified: graphs\05_compute_vs_year_scatter.png (162,144 bytes)
Verified: graphs\06_log10_compute_vs_year_scatter.png (336,226 bytes)
Verified: graphs\07_monthly_valid_compute_counts.png (193,852 bytes)
Verified: graphs\08_yearly_median_log10_compute.png (313,944 bytes)
Verified: graphs\09_monthly_median_log10_compute.png (550,104 bytes)
All 9 figures saved in graphs/ directory. (IPython display skipped in standard Python runtime)


### 10. Task 10 & 11: Statistical Frequency Determination & Historical Sparsity
We compare Yearly vs. Monthly frequency across objective criteria and determine the recommended modeling start year.

In [20]:
# Analyze annual data sparsity
full_years = range(int(df['Year'].min()), int(df['Year'].max()) + 1)
sparsity_records = []
for y in full_years:
    t_m = (df['Year'] == y).sum()
    v_c = (clean_compute_df['Year'] == y).sum()
    sparsity_records.append({'Year': y, 'Total_Models': t_m, 'Valid_Compute': v_c})
s_df = pd.DataFrame(sparsity_records)

zero_c_years = s_df[s_df['Valid_Compute'] == 0]['Year'].tolist()
low_c_years = s_df[(s_df['Valid_Compute'] > 0) & (s_df['Valid_Compute'] < 5)]['Year'].tolist()
strong_c_years = s_df[s_df['Valid_Compute'] >= 5]['Year'].tolist()

print(f'Years with Zero Compute Records ({len(zero_c_years)}): {zero_c_years}')
print(f'Years with Low Compute Records (1-4) ({len(low_c_years)}): {low_c_years}')
print(f'Years with Strong Compute Records (>=5) ({len(strong_c_years)}): {strong_c_years}')

total_pre_2010 = clean_compute_df[clean_compute_df['Year'] < 2010]['compute_numeric'].count()
total_post_2010 = clean_compute_df[clean_compute_df['Year'] >= 2010]['compute_numeric'].count()
total_post_2012 = clean_compute_df[clean_compute_df['Year'] >= 2012]['compute_numeric'].count()

print(f'\nPre-2010 Valid Compute Records: {total_pre_2010} ({total_pre_2010/len(clean_compute_df)*100:.2f}%)')
print(f'2010–2026 Valid Compute Records: {total_post_2010} ({total_post_2010/len(clean_compute_df)*100:.2f}%)')
print(f'2012–2026 Valid Compute Records: {total_post_2012} ({total_post_2012/len(clean_compute_df)*100:.2f}%)')

Years with Zero Compute Records (26): [1951, 1952, 1953, 1954, 1955, 1956, 1958, 1961, 1964, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1976, 1977, 1978, 1979, 1981, 1982, 1984, 1985, 2002]
Years with Low Compute Records (1-4) (32): [1950, 1957, 1959, 1960, 1962, 1963, 1965, 1966, 1975, 1980, 1983, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2003, 2004, 2005, 2006, 2007, 2008]
Years with Strong Compute Records (>=5) (19): [1994, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Pre-2010 Valid Compute Records: 62 (11.48%)
2010–2026 Valid Compute Records: 478 (88.52%)
2012–2026 Valid Compute Records: 468 (86.67%)


### 11. Task 12: Outlier Diagnostic (Raw vs. Log10)
We test for statistical outliers using IQR and Z-scores on raw compute and $\log_{10}$ compute. We examine why frontier models are genuine milestones rather than measurement errors.

In [22]:
# Raw compute IQR
q1_raw = clean_compute_df['compute_numeric'].quantile(0.25)
q3_raw = clean_compute_df['compute_numeric'].quantile(0.75)
iqr_raw = q3_raw - q1_raw
raw_outliers = clean_compute_df[clean_compute_df['compute_numeric'] > q3_raw + 1.5 * iqr_raw]
print(f'Raw Compute Outliers (Q3 + 1.5*IQR): {len(raw_outliers)}')

# Log10 compute IQR
q1_log = clean_compute_df['log10_compute'].quantile(0.25)
q3_log = clean_compute_df['log10_compute'].quantile(0.75)
iqr_log = q3_log - q1_log
lower_log = q1_log - 1.5 * iqr_log
upper_log = q3_log + 1.5 * iqr_log
log_low = clean_compute_df[clean_compute_df['log10_compute'] < lower_log]
log_high = clean_compute_df[clean_compute_df['log10_compute'] > upper_log]
print(f'log10 compute IQR bounds: [{lower_log:.3f}, {upper_log:.3f}]')
print(f'log10 Low Outliers: {len(log_low)}')
print(f'log10 High Outliers: {len(log_high)}')

# Display top compute systems
print('\nTop 5 Compute Systems in Dataset:')
clean_compute_df.sort_values(by='compute_numeric', ascending=False)[['Model', 'Organization', 'Year', 'compute_numeric', 'log10_compute', 'Frontier model']].head(5)

Raw Compute Outliers (Q3 + 1.5*IQR): 108
log10 compute IQR bounds: [9.424, 31.899]
log10 Low Outliers: 19
log10 High Outliers: 0

Top 5 Compute Systems in Dataset:


### 12. Task 16: Final Console Summary
Summary formatted according to user specifications.

In [24]:
print('''
DATASET SUMMARY
----------------
Rows: 1072
Columns: 47
Earliest date: 1950-07-02
Latest date: 2026-09-21
Valid FLOP records: 540
Missing FLOP records: 532 (49.63%)
Valid parameter records: 725
Year count: 62 observed years (77 calendar years)
Month count: 193 observed months (915 calendar months)

TIME-SERIES COVERAGE
--------------------
Years with compute: 51
Months with compute: 193
Months without compute: 722 (78.91% empty)
Recent-year coverage: 2020: 66.0%, 2021: 78.5%, 2022: 65.9%, 2023: 54.6%, 2024: 46.9%, 2025: 37.4%, 2026: 26.9% (partial)

RECOMMENDATION
--------------
Preferred frequency: YEARLY
Reason: Monthly data has 78.91% missing periods across full history (and 22% missing even in the modern era). With a median of only 2 models per observed month, monthly series exhibit extreme sampling variance and conference-cycle noise. In contrast, yearly data has 0% missing periods over the active modeling horizon (averaging 31.2 models/year), guaranteeing robust, defensible statistical aggregation.
Recommended modeling start year: 2010 (or 2012 landmark AlexNet era), holding partial 2026 for evaluation.
''')


DATASET SUMMARY
----------------
Rows: 1072
Columns: 47
Earliest date: 1950-07-02
Latest date: 2026-09-21
Valid FLOP records: 540
Missing FLOP records: 532 (49.63%)
Valid parameter records: 725
Year count: 62 observed years (77 calendar years)
Month count: 193 observed months (915 calendar months)

TIME-SERIES COVERAGE
--------------------
Years with compute: 51
Months with compute: 193
Months without compute: 722 (78.91% empty)
Recent-year coverage: 2020: 66.0%, 2021: 78.5%, 2022: 65.9%, 2023: 54.6%, 2024: 46.9%, 2025: 37.4%, 2026: 26.9% (partial)

RECOMMENDATION
--------------
Preferred frequency: YEARLY
Reason: Monthly data has 78.91% missing periods across full history (and 22% missing even in the modern era). With a median of only 2 models per observed month, monthly series exhibit extreme sampling variance and conference-cycle noise. In contrast, yearly data has 0% missing periods over the active modeling horizon (averaging 31.2 models/year), guaranteeing robust, defensible stat